In [0]:
%fs ls /Volumes/workspace/default/big_mart/

In [0]:
from pyspark.sql.functions import *
df_sel = spark.read.format('csv')\
                    .option('inferSchema',True)\
                    .option('header',True)\
                    .load('dbfs:/Volumes/workspace/default/big_mart/BigMart Sales.csv')

In [0]:
df_sel.display()

#Using select

DataFrame methods operate on the entire DataFrame (like select, filter, withColumn), while functions from pyspark.sql.functions operate on individual columns and must be used inside DataFrame methods.

In [0]:
df_sel.select('Item_Identifier','Item_Weight','Item_Fat_Content','Item_Type','Outlet_Size','Item_MRP').display()
#select is case sensitive, you cant use SELECT 

#Using col

In [0]:
from pyspark.sql.functions import col
#col needs to be imported to be used

In [0]:
df_sel.select(col("Item_Fat_Content"),col("Item_MRP"),col("Item_Outlet_Sales")).display()
#using col to select columns, useful in case you want to perform any kind of aggregations on the columns

###Alias

In [0]:
df_sel.select(col('Item_Identifier').alias('Item_Id')).display()
#you can use alias only on the col object and not on column names directly

#Using Filter

#### Scenario 1 filtering on single column

In [0]:
df_sel.filter(col('Item_Fat_Content')=='Regular').display()
#Using filer kinda similar to where clause in sql

#### Scenario 2: FIltering and using and on 2 columns

In [0]:
df_sel.filter((col("Item_MRP")<100)&(col("Item_Fat_Content")=='Low Fat')&(col('Item_Type')=='Dairy')).display()

In [0]:
df_sel.display()

####isin and isNull

In [0]:
df_sel.filter((col('Outlet_Location_Type').isin('Tier 1','Tier 2')) & (col('Outlet_Size').isNull())).display()
#isin is kinda like in and isNull is used for finding null values in a column

###withColumnRenamed
withColumnRenamed will rename the column on existing df and wont need a select function to be used. Meanwhile for alias we need to use it with .select or .agg function and it will create a new df instead of renaming column in original df

In [0]:
df_sel.withColumnRenamed('Item_Fat_Content','Fat_Content').display()

#withColumn

#### Scenario 1 Creating new column

In [0]:
#withColumn() creates a new DataFrame with a column added or replaced.
#Spark DataFrames are immutable, so it doesn't modify the original DataFrame in-place
# SYNTAX df.withColumn("column_name", column_expression), if same column_name used which is present in table then it'll modify existing column otherwise create new column.

In [0]:
from pyspark.sql.functions import lit

In [0]:
df = df_sel.withColumn('Flag',lit('Y'))
#lit() is used to create a literal (constant) column value in a DataFrame.

In [0]:
df.display()

In [0]:
from pyspark.sql.functions import concat

In [0]:
df.withColumn('Item_Key',concat(col('Item_Identifier'),col('Outlet_Identifier'))).display()

In [0]:
df.withColumn('Product',(col('Item_Weight')*col('Item_MRP'))).display()

####Modifying existing column

In [0]:
from pyspark.sql.functions import *

In [0]:
df_sel.withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),'Regular','Reg'))\
      .withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Low Fat","LF"))\
      .withColumn('Item_Type',regexp_replace(col('Item_Type')," ",""))\
      .display()
#regexp_replace It replaces text in a column using a regular expression pattern.
# SYNTAX regexp_replace(column, pattern, replacement)

#Type Casting

### cast() in PySpark is used to convert a column from one data type to another.
This is very common in ETL pipelines, especially when data comes from CSV/JSON where everything is read as string.

In [0]:
df = df_sel.withColumn('Item_Weight',col('Item_Weight').cast(StringType()))
#SYNTAX df.withColumn("column_name", col("column_name").cast("datatype"))
# df.select(
#    col("price").cast("int").alias("price_int")) Can also be used inside select function
# supported datatypes are: string,int,double,boolean,float,date,timestamp

In [0]:
df1 = df_sel.withColumn('Item_Visibility',col('Item_Visibility').cast('double'))

In [0]:
df.printSchema()

In [0]:
df1.printSchema()

#Sort

In PySpark, sort() is used to order rows in a DataFrame based on one or more columns.
It is similar to SQL ORDER BY.

In [0]:
df_sel.sort(col('Item_weight').desc()).display()
#SYNTAX df.sort(col("column_name")) by default it will sort ascending order
#       df,sort(col('col_name').desc()) this can be used for sorting in descending order

Sorting is expensive in distributed systems because Spark must shuffle data across partitions.

So avoid unnecessary sorts in large pipelines.

In [0]:
df_sel.sort(col('Item_Visibility').asc(),col('Item_MRP').desc()).display()

In [0]:
df_sel.sortWithinPartitions(col('Item_MRP').desc()).display()
# sortWithinPartitions will sort the data in each partition instead of doing a global sort in the df, mostly useful during the time of writing data. 

In [0]:
df_sel.sort(['Item_MRP','Item_Weight'],ascending=[0,0]).display()
#Another was top sort multiple colms simultaneously, [0,0] means false boolean so it will sort it as descending, if given 1 it will sort as ascending

#LIMIT

limit() in PySpark is used to return only the first N rows from a DataFrame.

Spark data is stored in multiple partitions, so limit() does not necessarily mean "first rows in sorted order".

It simply stops reading after N rows are collected. SO if a Partition A has 3 records and Partition B has 3 records then based limit(3), might show all 3 records from partition A or B, it basically depends on which partition gets processed first for this task if partition A gets processed first it'll show all 3 records from partition A and vice-versa.

In [0]:
df_sel.limit(100).display()
#

#Drop
used to drop one or more columns from a dataframe

In [0]:
df_sel.drop('Outlet_Identifier').display()

In [0]:
df_sel.drop("Item_MRP","Outlet_Size").display()

#Dropping Duplicates

dropDuplicates() in PySpark is used to remove duplicate rows from a DataFrame.

It works similarly to SQL DISTINCT, but it also allows you to specify specific columns to check for duplicates.

dropDuplicates() requires a shuffle because Spark must group rows across partitions.

Execution plan often includes:

Exchange
HashAggregate

Which means data redistribution across cluster.

In [0]:
df_sel.dropDuplicates().display()
#this is also called as dedup, it will drop all the duplicate records in the df

In [0]:
df_sel.dropDuplicates(subset=['Item_Type','Outlet_Type','Outlet_Location_Type']).display()
#Think of it like Spark creating a temporary hash key:
# hash(Item_Type, Outlet_Type, Outlet_Location_Type)
# Then it keeps one row per hash key.
# That’s why a shuffle happens — Spark must bring identical keys together

#Union and Union by Name

####Creating Dataframes

createDataFrame() is a SparkSession method used to create a Spark DataFrame from local data or other data structures like lists, RDDs, Pandas DataFrames, etc.

It is usually used when:

creating sample/test datasets

converting Pandas → Spark

converting RDD → DataFrame

**SYNTAX --> spark.createDataFrame(data, schema=None)**

In [0]:
data1=[(1,'KC'),(2,'CK'),(3,'Ash')]
schema = 'Id int, Name string'
df1= spark.createDataFrame(data1,schema)

In [0]:
data2 =[(4,'Aoi'),(5,'Z')]
df2 = spark.createDataFrame(data2,schema)

In [0]:
df1.display()

In [0]:
df2.display()

In [0]:
data3=[('KC',1),('CK',2),('Ash',3)]
schema = 'Name string, Id int'
df3= spark.createDataFrame(data3,schema)

In [0]:
df3.display()

In [0]:
data4=[('Zach'),('CuK'),('Asha')]
schema = 'Name string'
df4= spark.createDataFrame(data4,schema)

In [0]:
df4.display()

###Union

union() in PySpark is used to combine rows from two DataFrames (stack them vertically).
Both df should have same no of colms of same datatypes and in same order. in spark union will behave like union all in sql and will keep duplicate values. 

In [0]:
df2.union(df1).display()

In [0]:
df2.union(df1).sort(col('ID').desc()).display()

###UnionByName

unionByName() (very useful)

Spark also provides:

SYNTAX --> df1.unionByName(df2)

This matches columns by name instead of position.

In [0]:
df_new = df2.unionByName(df3).sort(col('Id').desc())
#Here by using unionByName, spark will make the union based on the columnnames instead. 

In [0]:
df_new.unionByName(df4,allowMissingColumns =True).display()
# by specifying allowMissingColumns = True, we can add columns that are not in the other dataframe and populate null for absent values. This is useful for unioning dataframes with different schemas.

#String Functions

#### initcap()

Capitalizes the first letter of each word and converts the rest to lowercase.

Syntax-->
initcap(col("column_name"))

In [0]:
df_sel.select(initcap('Item_Type')).display()

####lower()

Converts all characters in a string to lowercase.


SYNTAX -->lower(col("column_name"))



df.withColumn("name_lower", lower(col("name"))).show()

In [0]:
df_sel.select((lower('Item_Type')).alias('Low_IT')).display()

####upper()

Converts all characters to uppercase.

Syntax -->
upper(col("column_name"))

df.withColumn("name_upper", upper(col("name"))).show()

In [0]:
df_sel.select((upper('Item_Type')).alias('Cap_IT')).display()

#Date Functions

####CURRENT_DATE()

current_date() in PySpark returns the current system date (based on the Spark cluster/system time). It is commonly used in ETL pipelines, auditing, and data tracking.

It is a column function, so it must be imported.

####current_timestamp
adds date as well as time

In [0]:
df_d = df_sel.withColumn('Current_date',current_date())
  

####Date_Add()

date_add() in PySpark is used to add a number of days to a date column.

It is a column function, so you need to import it.

SYNTAX --> date_add(Column, days_to_add)

You can also subtract days using -ve sign

In [0]:
df_d.withColumn('Order_date',date_add(col('Current_date'),7)).display()

####Date_Sub()

date_sub() in PySpark is used to subtract a number of days from a date column.

Like date_add(), it is a column function, so you must import it.

SYNTAX --> date_sub(column, days_to_subtract)
You Can also add days using -ve sign

In [0]:
df_d.withColumn('W_before',date_sub(col('Current_date'),7)).display()

In [0]:
df_sel.withColumn('W_Before',date_sub(current_date(),7)).display()

####DATEDIFF

datediff() in PySpark is used to calculate the difference in days between two dates.

It returns an integer (number of days).

You must import it from pyspark.sql.functions.

SYNTAX --> datediff(end_date, start_date)

In [0]:
df_new=df_sel.withColumn('Current_date',current_date())\
             .withColumn('Week_After',date_add(col('Current_date'),7))\
             .withColumn('2Week_Before',date_sub(col('Current_date'),14))

In [0]:
df_new.display()

In [0]:
df_new.withColumn('Date_diff',datediff(col('Week_After'),col('Current_date'))).display()

####Date_Format

date_format() in PySpark is used to convert a date or timestamp column into a specific string format.

It is very common in reporting, dashboards, and data warehouse transformations.

You must import it from pyspark.sql.functions.

SYNTAX --> date_format(date_column, format)

Format	Example Output
- "yyyy-MM-dd"	2024-01-05
- "dd-MM-yyyy"	05-01-2024
- "MM/dd/yyyy"	01/05/2024
- "yyyy/MM/dd"	2024/01/05

| Format Symbol | Meaning      | Example |
| ------------- | ------------ | ------- |
| yyyy          | year         | 2024    |
| MM            | month number | 01      |
| MMM           | month short  | Jan     |
| MMMM          | month full   | January |
| dd            | day          | 05      |
| EEEE          | day name     | Friday  |


In [0]:
df_new.display()

In [0]:
df_new.withColumn('Current_date', date_format(col('Current_date'),'dd-MM-yyyy'))\
    .withColumn('Week_After',date_format(col('Week_After'),'dd-MM-yyyy'))\
    .withColumn('2Week_Before',date_format(col('2Week_Before'),'dd-MM-yyyy'))\
    .display()

In [0]:
df_new.withColumn('Day',date_format(col('Current_date'),'EEEE-dd-MMM-yyyy')).display()

# Handling Nulls

There are 2 ways for handling Nulls 
1. Dropping NullS
2. Filling Nulls

#### Dropping Nulls

dropna() in PySpark is used to remove rows containing NULL values from a DataFrame.

It is a DataFrame method, so you use it directly on the dataframe.

Note: It only removed Null rows and if there's a string called 'Null' it wont remove it.

SYNTAX --> df.dropna(how='any', thresh=None, subset=None)

| Parameter | Meaning                                    |
| --------- | ------------------------------------------ |
| `how`     | condition for dropping rows                |
| `thresh`  | minimum number of non-null values required |
| `subset`  | columns to check for nulls                 |


df.dropna(thresh=2)


Row must have at least 2 non-null values

i.e. if a row has only 1 non null value, it will get dropped from the table

| Usage                     | Meaning                            |
| ------------------------- | ---------------------------------- |
| `df.dropna()`             | drop rows with any null            |
| `df.dropna(how='all')`    | drop rows where all columns null   |
| `df.dropna(subset=[...])` | check null only in certain columns |
| `df.dropna(thresh=n)`     | require minimum non-null columns   |


In [0]:
df_sel.dropna('all').display()
#It will drop a record only if all the columns are null. If any one column is not null, it will not drop the record

In [0]:
df_sel.dropna('any').display()
#It will drop a record if any one column is null.

In [0]:
df_sel.dropna(subset=['Outlet_Size']).display()

In [0]:
dropped_rows = df_sel.count() - df_sel.dropna('any').count()
print(f'The no of rows dropped:{dropped_rows}' )

#### Filling nulls

fillna() in PySpark is used to replace NULL values with a specified value instead of removing the rows.

It is a DataFrame method, so it is used directly on the DataFrame.

SYNTAX -->df.fillna(value, subset=None)

| Parameter | Meaning                                 |
| --------- | --------------------------------------- |
| `value`   | value used to replace NULLs             |
| `subset`  | columns where replacement should happen |


| Usage                            | Meaning                                         |
| -------------------------------- | ----------------------------------------------- |
| `df.fillna(0)`                   | replace numeric nulls with 0                    |
| `df.fillna("Unknown")`           | replace string nulls                            |
| `df.fillna(value, subset=[...])` | replace nulls in specific columns               |
| `df.fillna({col:value})`         | replace different columns with different values |


In [0]:
df_sel.fillna('NotAvailable').display()

In [0]:
df_sel1= df_sel.withColumn('Item_Weight',col('Item_Weight').cast('string'))
df_sel1.fillna('NA').display()

In [0]:
df_sel.fillna('NotAvailable',subset=['Outlet_Size']).display()
#subset is used to tell spark onto which column it should the values in 

#Split and Indexing

#### Split

split() in PySpark is a string function used to split a string column into an array of values based on a delimiter or pattern.

It is very common when processing CSV-like strings, names, codes, or log data.

You must import it from pyspark.sql.functions.

split will return a list, e.g below

SYNTAX --> split(column, pattern)

| Function  | Purpose                 |
| --------- | ----------------------- |
| `split()` | break string into array |
| delimiter | defines split point     |
| result    | array column            |


In [0]:
df = df_sel.withColumn('Outlet_Type',split(col('Outlet_Type'),' '))

####Indexing

indexing usually refers to accessing elements from complex column types, most commonly arrays (like the result of split()).

Instead of [ ], Spark also provides: getItem() method

In [0]:
df.withColumn('Outlet_Type',col('Outlet_Type')[0]).display()

In [0]:
df.select(col('Outlet_Type').getItem(0)).display()

In [0]:
df.withColumn('Store_Type',col('Outlet_Type').getItem(1)).display()

#Array handling

#### Explode

explode() in PySpark is used to convert elements of an array (or map) into multiple rows.

It is extremely useful when working with arrays, JSON data, or results of split().

If a column contains an array, explode() will create one row per element.

Think of it like unpacking the array into separate rows

If the array is empty or NULL:

[]
NULL

explode() removes the row.

E.g.
| id | fruits                  |
| -- | ----------------------- |
| 1  | [apple, banana, orange] |

Result after explode
| id | fruit  |
| -- | ------ |
| 1  | apple  |
| 1  | banana |
| 1  | orange |



In [0]:
df.display()

In [0]:
df.withColumn('Outlet_Type',explode(col('Outlet_Type'))).display()

| Function             | Purpose                           |
| -------------------- | --------------------------------- |
| `explode()`          | array → multiple rows             |
| `explode_outer()`    | keeps NULL rows                   |
| `posexplode()`       | explode + index                   |
| `posexplode_outer()` | explode with index + NULL support |


In [0]:
df.select(posexplode(col('Outlet_Type'))).display()